In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader


In [2]:
import torchvision.transforms as transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5) , (0.5,0.5,0.5))
])
trainset = CIFAR10(root = "./data", train = True , download = False , transform = transform)
testset = CIFAR10(root = "./data" , train = False , download = True , transform = transform)
 

In [3]:
trainLoader = DataLoader(trainset , batch_size = 64 , shuffle = True)
testLoader = DataLoader(testset , batch_size = 64)

In [4]:
class CNN(nn.Module) :
    def __init__(self) :
        super(CNN , self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32, kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64, kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128 , 256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [5]:
model = CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [9]:
epochs = 20
for epoch in range(epochs) :
    epoch_training_loss = 0.0

    for images , labels in trainLoader :
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output,labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss /len(trainLoader)}") 

epoch = 1/20 & loss = 0.12861524575420885
epoch = 2/20 & loss = 0.11080357086156374
epoch = 3/20 & loss = 0.09784817847046916
epoch = 4/20 & loss = 0.08715671765477494
epoch = 5/20 & loss = 0.08398790665380561
epoch = 6/20 & loss = 0.0843248329345075
epoch = 7/20 & loss = 0.07080018780399067
epoch = 8/20 & loss = 0.0788890335370478
epoch = 9/20 & loss = 0.07014123770846602
epoch = 10/20 & loss = 0.0632749903892629
epoch = 11/20 & loss = 0.07390855269862667
epoch = 12/20 & loss = 0.05839017700210995
epoch = 13/20 & loss = 0.05805501944291384
epoch = 14/20 & loss = 0.06499744049759339
epoch = 15/20 & loss = 0.0706229432253167
epoch = 16/20 & loss = 0.05484340465454923
epoch = 17/20 & loss = 0.05031770047096177
epoch = 18/20 & loss = 0.059890103590129244
epoch = 19/20 & loss = 0.05867620723628257
epoch = 20/20 & loss = 0.04882468816781383


In [12]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad() :
    for images , labels in testLoader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs , 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

    print(f"accuracy = {correct_labels / total_labels * 100 }")

accuracy = 74.83
